# Lesson 36: MCP, agent-to-agent communication and agent frameworks 🔌🤝
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Explain the **Model Context Protocol (MCP)**: hosts, clients, servers, tools, resources and prompts
2. Build an **MCP server**, connect to it with a **client**, and let an LLM use its tools
3. Explain **agent-to-agent (A2A)** communication: agent cards, tasks and messages
4. Compare popular **agent frameworks** and choose one for a project

**⏱️ Timing:** MCP idea 10' | build a server 15' | LLM + MCP 15' | A2A 10' | frameworks 10'

> ⚙️ **GPU required:** Runtime → Change runtime type → **T4 GPU**. Agents need a smarter model, so this lesson uses `qwen2.5:7b` (about 5 GB, 2–3 minutes to download). If that's too slow, change it to `qwen2.5:3b` — it works, but makes more tool-calling mistakes.

## 🔌 An analogy: USB-C for AI
Before USB, every device needed its own special cable. Before MCP, every AI app needed its own special code for every tool (Google Drive, GitHub, a database…).

The **Model Context Protocol** (introduced by Anthropic in November 2024, now an open standard used across many AI apps) is a **shared plug**: build a tool server **once**, and any MCP-compatible AI app can use it.

| Part | Role | Example |
|---|---|---|
| **Host** | The AI app the user talks to | Claude Desktop, an IDE, your own Python app |
| **Client** | Lives inside the host; talks to one server | `fastmcp.Client` |
| **Server** | Offers capabilities | A "school timetable" server |
| **Tools** | Actions the model can call | `get_homework(subject)` |
| **Resources** | Read-only data the app can load | `school://rules` |
| **Prompts** | Reusable prompt templates | "Make a study plan for {subject}" |

**MCP vs A2A:** MCP connects an agent to its **tools and data** (vertical). **A2A** connects **agents to other agents** (horizontal).

In [ ]:
# 🦙 Start a free LLM server *inside* Colab (Ollama). No API key or account needed.
# ⏳ First run: about 2–3 minutes (installs Ollama and downloads the model).
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

MODEL = "qwen2.5:7b"     # smarter at using tools. "qwen2.5:3b" downloads faster but makes more mistakes
!ollama pull {MODEL} > /dev/null 2>&1
print("✅ Ollama is running with", MODEL)

In [ ]:
!pip -q install fastmcp "fastmcp-slim[client]" openai
import json, asyncio, warnings; warnings.filterwarnings("ignore")
print("ready")

## Part 1: Build an MCP server 🏫
We'll write the server to a file and run it in the background, just like a real server.

In [ ]:
%%writefile school_server.py
from fastmcp import FastMCP

mcp = FastMCP("School Helper")

TIMETABLE = {"monday": "Maths, English, Science, PE", "tuesday": "History, Maths, Art, Music",
             "wednesday": "Science, English, Te Reo Māori, Maths", "thursday": "Geography, Science, Maths, Drama",
             "friday": "English, Technology, Health, Assembly"}
HOMEWORK = {"maths": "Algebra worksheet 4, due Friday", "science": "Write up the pendulum experiment, due Wednesday",
            "english": "Read chapter 6 of the class novel"}

@mcp.tool
def get_timetable(day: str) -> str:
    """Get the list of classes for a school day (monday to friday)."""
    return TIMETABLE.get(day.lower(), "No school that day")

@mcp.tool
def get_homework(subject: str) -> str:
    """Get the current homework for a subject."""
    return HOMEWORK.get(subject.lower(), "No homework set")

@mcp.resource("school://rules")
def rules() -> str:
    """The school's top rules."""
    return "1. Be kind. 2. Be on time. 3. Phones away in class."

@mcp.prompt
def study_plan(subject: str) -> str:
    """A prompt template for a study plan."""
    return f"Make a 3-day study plan for {subject} for a 15-year-old. Use short bullet points."

if __name__ == "__main__":
    mcp.run(transport="http", host="127.0.0.1", port=8000)

In [ ]:
import subprocess, time
server = subprocess.Popen(["python", "school_server.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(6)
print("🟢 MCP server running at http://127.0.0.1:8000/mcp")

## Part 2: Connect with an MCP client 🔗
In Colab you can use `await` directly in a cell.

In [ ]:
from fastmcp import Client
SERVER = "http://127.0.0.1:8000/mcp"

async with Client(SERVER) as c:
    tools = await c.list_tools()
    print("🛠️ TOOLS:")
    for t in tools: print(f"   {t.name}: {t.description}  inputs={list(t.input_schema['properties'])}")
    print("📄 RESOURCES:", [str(r.uri) for r in await c.list_resources()])
    print("💬 PROMPTS:", [p.name for p in await c.list_prompts()])

    result = await c.call_tool("get_timetable", {"day": "Wednesday"})
    print("\nget_timetable('Wednesday') →", result.data)
    rules = await c.read_resource("school://rules")
    print("school://rules →", rules[0].text)

## Part 3: Let an LLM use the MCP server 🤖
The agent doesn't know these tools in advance. It **discovers** them from the server, converts them to the tool format the LLM understands (Lesson 35), and calls them through MCP. Add a new tool to the server, and the agent can use it with **no changes** to the agent code.

In [ ]:
from openai import OpenAI
llm = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

async def mcp_agent(question, max_steps=6):
    async with Client(SERVER) as c:
        tools = await c.list_tools()
        specs = [{"type": "function", "function": {"name": t.name, "description": t.description, "parameters": t.input_schema}}
                 for t in tools]
        messages = [{"role": "system", "content": "You help students. Use the school tools for timetable and homework facts."},
                    {"role": "user", "content": question}]
        for _ in range(max_steps):
            reply = llm.chat.completions.create(model=MODEL, messages=messages, tools=specs, temperature=0).choices[0].message
            messages.append(reply.model_dump(exclude_none=True))
            if not reply.tool_calls:
                return reply.content
            for call in reply.tool_calls:
                args = json.loads(call.function.arguments or "{}")
                try:
                    result = (await c.call_tool(call.function.name, args)).data
                except Exception as e:                      # report mistakes back to the model instead of crashing
                    result = f"error: {e}"
                print(f"  🔌 MCP → {call.function.name}({args}) = {result}")
                messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})
    return "Stopped."

print("✅", await mcp_agent("What classes do I have on Thursday, and what's my maths homework?"))

### 🧩 Exercise 1
1. Add a new tool to `school_server.py`, e.g. `get_canteen_menu(day)`. Restart the server (run `server.kill()`, then the start cell again) and ask about lunch. Did you change the agent code? (No — that's the point of MCP!)
2. Use the `study_plan` prompt: `await c.get_prompt("study_plan", {"subject": "science"})`, then send the resulting messages to the LLM.
3. **Security:** an MCP server can do anything its code allows. Why should you only connect to MCP servers you trust? What if a tool's *description* contained hidden instructions ("tool poisoning")?

## Part 4: Agent-to-agent (A2A) communication 🤝
**A2A** (the Agent2Agent protocol, introduced by Google in April 2025 and now an open project under the Linux Foundation) lets agents from **different companies and frameworks** work together. Key ideas:
- **Agent Card** — a public "business card" (JSON) describing what an agent can do and how to reach it (usually at `/.well-known/agent-card.json`)
- **Task** — a unit of work with a status: `submitted → working → completed / failed / input-required`
- **Messages and artifacts** — what agents send each other (text, files, data)

Below is a simplified simulation of the idea in plain Python (real A2A uses HTTP and JSON-RPC).

In [ ]:
agent_cards = [
    {"name": "TimetableAgent", "description": "Knows class timetables and homework.", "skills": ["timetable", "homework"]},
    {"name": "MathsTutorAgent", "description": "Explains maths problems step by step.", "skills": ["algebra", "arithmetic", "geometry"]},
    {"name": "TranslatorAgent", "description": "Translates short texts between languages.", "skills": ["translation"]},
]

def ask_llm(prompt, system="Be brief."):
    return llm.chat.completions.create(model=MODEL, temperature=0, messages=[
        {"role": "system", "content": system}, {"role": "user", "content": prompt}]).choices[0].message.content.strip()

async def remote_agent(name, message):
    """Pretend each agent is a separate service run by a different team."""
    if name == "TimetableAgent":
        return await mcp_agent(message)                     # this agent uses MCP tools internally!
    system = {"MathsTutorAgent": "You are a maths tutor. Explain briefly.", "TranslatorAgent": "You translate text. Output only the translation."}[name]
    return ask_llm(message, system)

async def coordinator(request):
    cards = json.dumps(agent_cards, indent=1)
    choice = ask_llm(f"Agent cards:\n{cards}\n\nWhich agent should handle this request? Reply with the agent name only.\nRequest: {request}")
    name = next((c["name"] for c in agent_cards if c["name"].lower() in choice.lower()), agent_cards[0]["name"])
    task = {"id": f"task-{abs(hash(request)) % 10000}", "to": name, "status": "submitted", "message": request}
    print("📨", json.dumps(task))
    task["status"] = "working"
    task["artifact"] = await remote_agent(name, request)
    task["status"] = "completed"
    print("📬", json.dumps({k: task[k] for k in ["id", "to", "status"]}))
    return task["artifact"]

for r in ["What's my science homework?", "How do I find the area of a triangle?", "Translate 'Good morning, class' into Spanish."]:
    print(f"🧑 {r}\n🤖 {await coordinator(r)}\n")

### 🧩 Exercise 2
1. Write an agent card for a new **QuizMakerAgent** and implement it. Does the coordinator route to it correctly?
2. Notice how **TimetableAgent** uses **MCP** inside, while agents talk to each other with **A2A**-style tasks. Draw the whole system as a diagram.

## Part 5: Agent frameworks 🧰
You've built agents by hand. Frameworks give you tested pieces for loops, tools, memory, multi-agent teamwork, tracing and deployment.
| Framework | From | Style | Good for |
|---|---|---|---|
| **LangGraph** (Lesson 34) | LangChain | Graphs of nodes and edges | Precise control, complex workflows |
| **CrewAI** (Lessons 37–39) | CrewAI | Teams of role-playing agents with tasks | Multi-agent "crews", fast to start |
| **LlamaIndex agents** (Lesson 40) | LlamaIndex | Agents over your data | Agentic RAG |
| **OpenAI Agents SDK** | OpenAI | Lightweight agents + handoffs | OpenAI-centred apps |
| **Claude Agent SDK** | Anthropic | The agent loop behind Claude Code | Coding and computer-task agents |
| **Google ADK** | Google | Agent Development Kit | Gemini / Google Cloud apps |
| **Microsoft Agent Framework / AutoGen** | Microsoft | Conversational multi-agent | Research, Azure apps |
| **smolagents** | Hugging Face | Agents that write code to act | Open models, small and simple |
| **n8n, Langflow** (Lesson 41) | — | Visual, drag-and-drop | No-code automation |

> ⚠️ This area changes monthly. Check each project's docs before you start a new project.

**How to choose:** How much control do you need? Which model provider? Do you need multiple agents? Is there a visual builder for non-programmers? Is it open source? Does it support MCP?

In [ ]:
server.kill()
print("🔴 MCP server stopped")

## 📌 Summary
- **MCP** = a universal plug between AI apps and tools/data: **servers** offer **tools, resources and prompts**; **clients** inside **hosts** use them
- Agents can **discover** MCP tools at runtime
- **A2A** = agents talking to agents using **agent cards**, **tasks** and **messages**
- Many **frameworks** exist; they package the same ideas you built by hand